# DeBERTa-v3-base: BABE-only vs Wikipedia-augmented training

This notebook runs the **new experiment** requested:

- **Condition A:** DeBERTa-v3-base trained on BABE only.
- **Condition B:** DeBERTa-v3-base trained on BABE + 4,000 balanced Wikipedia-derived MBIB examples.
- Both conditions use the **same saved BABE train/validation/test splits** and the same training settings.
- Checkpoints are selected using BABE validation macro-F1.
- Test metrics and predictions are saved for later comparison.

## Important comparability note
Your teammate's reported DeBERTa result used a 528-example test set. The saved BABE split used in this notebook has 508 test examples. Therefore, results from this notebook are **not a direct comparison** with your teammate's 82.20% accuracy unless the split examples are verified to be identical. Do not replace one result with the other.

The Wikipedia-derived labels are distant supervision and are not identical to expert BABE news-bias labels. The augmentation may help, do nothing, or hurt. The experiment measures that effect rather than assuming improvement.

**Runtime:** Google Colab with a T4 GPU recommended. Mount Google Drive so checkpoints and outputs persist after runtime reset.

## 1. Install dependencies

In [1]:
%pip install -q transformers datasets accelerate scikit-learn sentencepiece pandas==2.2.3

## 2. Mount Drive and set reproducible configuration

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, random, shutil, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer,
    EarlyStoppingCallback, set_seed
)

SEED = 42
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256
LABEL2ID = {"Non-biased": 0, "Biased": 1}
ID2LABEL = {0: "Non-biased", 1: "Biased"}

SPLIT_DIR = Path("/content/drive/MyDrive/NewsBias_Project_Backup/babe_sg2_splits")
DRIVE_ROOT = Path("/content/drive/MyDrive/NewsBias_Project_Backup/deberta_augmented_experiment")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 3. Load the existing BABE splits — do not regenerate them

In [2]:
assert all((SPLIT_DIR / f"{name}.csv").exists()
           for name in ["train", "validation", "test"]), (
    "Expected /content/babe_sg2_splits/{train,validation,test}.csv. "
    "Copy your saved split folder from Drive into /content/babe_sg2_splits first."
)

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

def normalize_binary_label(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int, np.integer, float, np.floating)) and v in (0, 1):
        return int(v)
    s = str(v).strip().lower().replace("_", "-")
    mapping = {
        "0": 0, "non-biased": 0, "non biased": 0, "nonbiased": 0, "unbiased": 0,
        "1": 1, "biased": 1
    }
    return mapping.get(s, np.nan)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    if "label" not in frame.columns and "label_bias" in frame.columns:
        frame["label"] = frame["label_bias"].map(normalize_binary_label)
    else:
        frame["label"] = frame["label"].map(normalize_binary_label)
    assert frame["label"].notna().all(), f"Unexpected/missing labels in {name}"
    frame["label"] = frame["label"].astype(int)
    frame["text"] = frame["text"].astype(str).str.strip()
    assert frame["text"].str.len().gt(0).all()

for name, frame in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(frame)} rows | labels={frame.label.value_counts().sort_index().to_dict()}")
print("Note: expected split sizes are 2579 / 550 / 508.")


Train: 2579 rows | labels={0: 1327, 1: 1252}
Validation: 550 rows | labels={0: 266, 1: 284}
Test: 508 rows | labels={0: 252, 1: 256}
Note: expected split sizes are 2579 / 550 / 508.


## 4. Load MBIB linguistic-bias split and audit source IDs

In [3]:
# The MBIB repo exposes 'linguistic_bias' as a named split, not as a config.
mbib_ds = load_dataset("mediabiasgroup/mbib-base", split="linguistic_bias")
mbib = mbib_ds.to_pandas()
print("Rows:", len(mbib))
print("Columns:", mbib.columns.tolist())
assert {"text", "label", "dataset_id"}.issubset(mbib.columns), (
    "MBIB schema differs from expected; inspect the dataset card before proceeding."
)
display(pd.crosstab(mbib["dataset_id"], mbib["label"], margins=True))


README.md:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

cognitive-bias.csv:   0%|          | 0.00/851k [00:00<?, ?B/s]

fake-news.csv:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

gender-bias.csv:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

mbib-aggregated/hate-speech.csv: reconstructing file:   0%|          |  0.00B / 99.0MB            

...

Generating political_bias split:   0%|          | 0/17704 [00:00<?, ? examples/s]

Generating racial_bias split:   0%|          | 0/9788 [00:00<?, ? examples/s]

Generating text_level_bias split:   0%|          | 0/9018 [00:00<?, ? examples/s]

Rows: 401862
Columns: ['id', 'text', 'label', 'dataset_id']


label,0,1,All
dataset_id,,,
3,10068,1662,11730
9,0,1444,1444
10,1862,1618,3480
26,178228,160833,339061
35,5842,28731,34573
38,37,683,720
75,4894,5960,10854
All,200931,200931,401862


## 5. Prepare the same 4,000-example balanced Wikipedia-derived sample

In [4]:
# Use only the Wikipedia-derived source IDs used in the previous experiment:
# 3 = WikiNPOV; 26 = Wiki Neutrality Corpus.
wiki = mbib[mbib["dataset_id"].astype(str).isin(["3", "26"])].copy()
wiki["text"] = wiki["text"].astype(str).str.strip()
wiki["label"] = pd.to_numeric(wiki["label"], errors="coerce")
wiki = wiki.dropna(subset=["text", "label"])
wiki = wiki[wiki["label"].isin([0, 1])].copy()
wiki["label"] = wiki["label"].astype(int)
wiki = wiki[wiki["text"].str.len().between(20, 1500)].copy()
wiki["text_norm"] = wiki["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
wiki = wiki.drop_duplicates("text_norm")

# Remove exact normalized text overlap with all BABE splits.
babe_texts = set(
    pd.concat([train_df["text"], val_df["text"], test_df["text"]])
    .astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)
wiki = wiki[~wiki["text_norm"].isin(babe_texts)]

MAX_PER_LABEL = 2000
sample_parts = []
for label in [0, 1]:
    part = wiki[wiki["label"] == label]
    n = min(MAX_PER_LABEL, len(part))
    sample_parts.append(part.sample(n=n, random_state=SEED + label))
wiki_sample = pd.concat(sample_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Wikipedia sample rows:", len(wiki_sample))
print("Label counts:", wiki_sample["label"].value_counts().sort_index().to_dict())
display(wiki_sample.groupby(["dataset_id", "label"]).size().rename("n").reset_index())
assert len(wiki_sample) == 4000, (
    "The current MBIB sample did not yield exactly 4,000 rows. "
    "Review the counts above and adjust MAX_PER_LABEL only before training."
)
wiki_sample.drop(columns=["text_norm"], errors="ignore").to_csv(
    DRIVE_ROOT / "wiki_augmentation_sample.csv", index=False
)


Wikipedia sample rows: 4000
Label counts: {0: 2000, 1: 2000}


,dataset_id,label,n
0,3,0,103
1,3,1,18
2,26,0,1897
3,26,1,1982


## 6. Tokenize both conditions

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def prepare_dataset(frame):
    work = frame[["text", "label"]].copy().reset_index(drop=True)
    work["text"] = work["text"].astype(str)
    work["labels"] = work["label"].astype(int)
    ds = Dataset.from_pandas(work[["text", "labels"]], preserve_index=False)
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    return ds.map(tokenize, batched=True, remove_columns=["text"])

train_babe_ds = prepare_dataset(train_df)
train_aug_frame = pd.concat([
    train_df[["text", "label"]],
    wiki_sample[["text", "label"]]
], ignore_index=True)
train_aug_ds = prepare_dataset(train_aug_frame)
val_ds = prepare_dataset(val_df)
test_ds = prepare_dataset(test_df)

print("BABE-only train rows:", len(train_babe_ds))
print("Augmented train rows:", len(train_aug_ds))
print("Validation rows:", len(val_ds), "| Test rows:", len(test_ds))


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/2579 [00:00<?, ? examples/s]

Map:   0%|          | 0/6579 [00:00<?, ? examples/s]

Map:   0%|          | 0/550 [00:00<?, ? examples/s]

Map:   0%|          | 0/508 [00:00<?, ? examples/s]

BABE-only train rows: 2579
Augmented train rows: 6579
Validation rows: 550 | Test rows: 508


## 7. Define the stable DeBERTa training procedure

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(labels, pred, average="macro", zero_division=0)
    }

def train_and_evaluate(condition, train_ds):
    output_dir = f"/content/deberta_{condition}"
    persistent_dir = DRIVE_ROOT / condition
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    # Fresh model initialization for each condition.
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id={"Non-biased": 0, "Biased": 1},
        torch_dtype=torch.float32
    )

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=1e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=4,
        weight_decay=0.01,
        warmup_steps=100,
        fp16=False,
        bf16=False,
        gradient_checkpointing=False,
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
        data_seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
    )

    trainer.train()
    print(f"{condition}: best checkpoint =", trainer.state.best_model_checkpoint)
    print(f"{condition}: best validation macro-F1 =", trainer.state.best_metric)

    # Test evaluation happens after checkpoint selection.
    output = trainer.predict(test_ds)
    y_true = output.label_ids
    y_pred = np.argmax(output.predictions, axis=-1)
    metrics = {
        "condition": condition,
        "model": MODEL_NAME,
        "best_validation_macro_f1": float(trainer.state.best_metric),
        "test_size": int(len(y_true)),
        "test_accuracy": float(accuracy_score(y_true, y_pred)),
        "test_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "test_nonbiased_f1": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "test_biased_f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "test_biased_precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "test_biased_recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0))
    }

    print("\\nClassification report:")
    print(classification_report(y_true, y_pred, labels=[0,1],
                                target_names=["Non-biased", "Biased"],
                                digits=4, zero_division=0))
    print("Confusion matrix (true rows, predicted columns):")
    print(confusion_matrix(y_true, y_pred, labels=[0,1]))

    # Save the selected model and tokenizer persistently in Google Drive.
    if persistent_dir.exists():
        shutil.rmtree(persistent_dir)
    persistent_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(persistent_dir / "best_model"))
    tokenizer.save_pretrained(str(persistent_dir / "best_model"))

    pred_df = test_df.copy().reset_index(drop=True)
    pred_df["true_label_id"] = y_true
    pred_df["predicted_label_id"] = y_pred
    pred_df["predicted_label"] = [ID2LABEL[int(x)] for x in y_pred]
    pred_df.to_csv(persistent_dir / "test_predictions.csv", index=False)
    pd.DataFrame([metrics]).to_csv(persistent_dir / "metrics.csv", index=False)

    del trainer, model, output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics

print("Training settings: FP32, lr=1e-5, batch=2, grad accumulation=8, max 4 epochs.")


Training settings: FP32, lr=1e-5, batch=2, grad accumulation=8, max 4 epochs.


## 8. Run Condition A — DeBERTa trained on BABE only

In [7]:
# This is a fresh reproduction on the saved 508-example test split.
result_babe_only = train_and_evaluate("babe_only", train_babe_ds)
print(result_babe_only)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

babe_only: best checkpoint = /content/deberta_babe_only/checkpoint-648
babe_only: best validation macro-F1 = 0.8308638015468606


\nClassification report:
              precision    recall  f1-score   support

  Non-biased     0.8210    0.8373    0.8291       252
      Biased     0.8367    0.8203    0.8284       256

    accuracy                         0.8287       508
   macro avg     0.8288    0.8288    0.8287       508
weighted avg     0.8289    0.8287    0.8287       508

Confusion matrix (true rows, predicted columns):
[[211  41]
 [ 46 210]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'condition': 'babe_only', 'model': 'microsoft/deberta-v3-base', 'best_validation_macro_f1': 0.8308638015468606, 'test_size': 508, 'test_accuracy': 0.8287401574803149, 'test_macro_f1': 0.8287394938445263, 'test_nonbiased_f1': 0.8290766208251473, 'test_biased_f1': 0.8284023668639053, 'test_biased_precision': 0.8366533864541833, 'test_biased_recall': 0.8203125}


## 9. Run Condition B — DeBERTa trained on BABE + Wikipedia-derived data

In [8]:
result_augmented = train_and_evaluate("babe_plus_wikipedia", train_aug_ds)
print(result_augmented)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,4.436795,0.452251,0.803636,0.803262
2,4.020862,0.417125,0.810909,0.810606
3,3.246991,0.500372,0.812727,0.812205
4,2.854960,0.517467,0.805455,0.805377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

babe_plus_wikipedia: best checkpoint = /content/deberta_babe_plus_wikipedia/checkpoint-1236
babe_plus_wikipedia: best validation macro-F1 = 0.8122051720651464


\nClassification report:
              precision    recall  f1-score   support

  Non-biased     0.8478    0.7738    0.8091       252
      Biased     0.7950    0.8633    0.8277       256

    accuracy                         0.8189       508
   macro avg     0.8214    0.8185    0.8184       508
weighted avg     0.8212    0.8189    0.8185       508

Confusion matrix (true rows, predicted columns):
[[195  57]
 [ 35 221]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'condition': 'babe_plus_wikipedia', 'model': 'microsoft/deberta-v3-base', 'best_validation_macro_f1': 0.8122051720651464, 'test_size': 508, 'test_accuracy': 0.8188976377952756, 'test_macro_f1': 0.8184219932553188, 'test_nonbiased_f1': 0.8091286307053942, 'test_biased_f1': 0.8277153558052435, 'test_biased_precision': 0.7949640287769785, 'test_biased_recall': 0.86328125}


## 10. Compare both conditions

In [9]:
comparison = pd.DataFrame([result_babe_only, result_augmented])
display(comparison)

delta = result_augmented["test_macro_f1"] - result_babe_only["test_macro_f1"]
delta_acc = result_augmented["test_accuracy"] - result_babe_only["test_accuracy"]
print(f"Augmented minus BABE-only test macro-F1: {delta:+.4f} ({delta*100:+.2f} percentage points)")
print(f"Augmented minus BABE-only test accuracy: {delta_acc:+.4f} ({delta_acc*100:+.2f} percentage points)")

comparison.to_csv(DRIVE_ROOT / "deberta_condition_comparison.csv", index=False)
with open(DRIVE_ROOT / "experiment_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "train_rows_babe_only": len(train_babe_ds),
        "train_rows_augmented": len(train_aug_ds),
        "validation_rows": len(val_ds),
        "test_rows": len(test_ds),
        "augmentation_rows": len(wiki_sample),
        "augmentation_source_ids": sorted(wiki_sample["dataset_id"].astype(str).unique().tolist()),
        "note": "Wikipedia labels are distant supervision; results are not directly comparable with teammate's 528-example test result unless split identity is verified."
    }, f, indent=2)
print("Saved comparison and configuration to:", DRIVE_ROOT)


,condition,model,best_validation_macro_f1,test_size,test_accuracy,test_macro_f1,test_nonbiased_f1,test_biased_f1,test_biased_precision,test_biased_recall
0,babe_only,microsoft/deberta-v3-base,0.830864,508,0.828740,0.828739,0.829077,0.828402,0.836653,0.820312
1,babe_plus_wikipedia,microsoft/deberta-v3-base,0.812205,508,0.818898,0.818422,0.809129,0.827715,0.794964,0.863281


Augmented minus BABE-only test macro-F1: -0.0103 (-1.03 percentage points)
Augmented minus BABE-only test accuracy: -0.0098 (-0.98 percentage points)
Saved comparison and configuration to: /content/drive/MyDrive/NewsBias_Project_Backup/deberta_augmented_experiment


In [10]:
from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/NewsBias_Project_Backup/"
    "deberta_augmented_experiment"
)

for condition in ["babe_only", "babe_plus_wikipedia"]:
    path = folder / condition

    print(f"\n--- {condition} ---")
    print("Metrics:", (path / "metrics.csv").exists())
    print("Predictions:", (path / "test_predictions.csv").exists())
    print("Model:", (path / "best_model").exists())

    if (path / "metrics.csv").exists():
        display(pd.read_csv(path / "metrics.csv"))


--- babe_only ---
Metrics: True
Predictions: True
Model: True


,condition,model,best_validation_macro_f1,test_size,test_accuracy,test_macro_f1,test_nonbiased_f1,test_biased_f1,test_biased_precision,test_biased_recall
0,babe_only,microsoft/deberta-v3-base,0.830864,508,0.82874,0.828739,0.829077,0.828402,0.836653,0.820312



--- babe_plus_wikipedia ---
Metrics: True
Predictions: True
Model: True


,condition,model,best_validation_macro_f1,test_size,test_accuracy,test_macro_f1,test_nonbiased_f1,test_biased_f1,test_biased_precision,test_biased_recall
0,babe_plus_wikipedia,microsoft/deberta-v3-base,0.812205,508,0.818898,0.818422,0.809129,0.827715,0.794964,0.863281


## 11. Interpretation and reporting checklist

- Report the measured test accuracy, macro-F1, class-specific F1, precision/recall, and confusion matrix.
- Do not claim augmentation improved the model unless the results support it.
- A small difference on one 508-example test set may be noise; use paired bootstrap confidence intervals or repeat training seeds before strong claims.
- This test set has already been used in earlier experiments, so treat these comparisons as exploratory unless a new locked article-disjoint test set is prepared.
- Your teammate's reported DeBERTa score used 528 test examples; it must not be presented as a direct baseline for this notebook unless exact test examples and labels match.
- Preserve all artifacts in `MyDrive/NewsBias_Project_Backup/deberta_augmented_experiment`.
